In [1]:
# ════════════════════════════════════════════════════════════════════════════
#  UNIVERSE REFRESHER NOTEBOOK
#  Rescreens annually for weak performers and surfaces replacements using:
#    - Trailing 12-month Sharpe (365-day basis, consistent with main notebook)
#    - Piotroski F-Score, CapEx/OCF, Asset Turnover,
#      Altman Distress Flag, Altman Z-Score
#    - GICS sector constraint: replacements must match the sector of the
#      stock being replaced
#  Candidate pool: S&P 500 + NASDAQ 100, then mid/large cap if needed
# ════════════════════════════════════════════════════════════════════════════


# ── Cell 1: Imports & Configuration ─────────────────────────────────────────

import numpy as np
import pandas as pd
import yfinance as yf
import requests
import warnings
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

# ── Parameters ───────────────────────────────────────────────────────────────

# Bottom N by trailing Sharpe are flagged as weak and eligible for swap
WEAK_N = 10

# Number of replacement candidates to surface per weak stock
TOP_CANDIDATES = 5

# Sharpe is computed on 365-day basis to match the main notebook
ADPY = 365

# Lookback window for trailing Sharpe (calendar days)
SHARPE_LOOKBACK_DAYS = 365

# Minimum trading days required in lookback window to compute a valid Sharpe
MIN_TRADING_DAYS = 200

# Minimum average daily dollar volume for a candidate to be liquid enough
MIN_AVG_DOLLAR_VOL = 5_000_000   # $5M/day

# Piotroski F-Score minimum to pass the fundamental screen
MIN_PIOTROSKI = 5                 # out of 9; >= 5 = financially healthy

# Altman Z-Score threshold — below this = distress zone
ALTMAN_SAFE_ZONE = 2.99           # > 2.99 = safe; 1.81–2.99 = grey; < 1.81 = distress

# ── Your current universe from the main notebook ─────────────────────────────
# ETFs and crypto are excluded from the swap candidates (they have no
# Piotroski score) but are still evaluated for weakness so you know
# which slots are underperforming.

CURRENT_UNIVERSE = [
    'SMH', 'GDX', 'DGRW', 'DXJ', 'BOTZ', 'LIT', 'URNM', 'ARKK', 'ARKG',
    'HACK', 'BLOK', 'QTUM', 'AIRR', 'IRBO', 'MTUM', 'DBMF', 'KMLM', 'GLD',
    'SPY', 'QQQ', 'TQQQ', 'BTC-USD', 'SOL-USD', 'ETH-USD', 'BNB-USD',
    'MSFT', 'AAPL', 'NVDA', 'XLE', 'AMLP', 'XLF', 'KRE', 'XLV', 'XBI',
    'XLI', 'XLB', 'XLP', 'XLY', 'VNQ', 'XLRE', 'TLT', 'HYG', 'EFA',
    'VGK', '^NSEI',
]

# Assets that are ETFs, crypto, or indices — they will be flagged for weakness
# but NO fundamental screen is run on them, and they cannot be replaced by
# individual stocks (you would replace them with a similar ETF manually).
NON_EQUITY_ASSETS = {
    'SMH', 'GDX', 'DGRW', 'DXJ', 'BOTZ', 'LIT', 'URNM', 'ARKK', 'ARKG',
    'HACK', 'BLOK', 'QTUM', 'AIRR', 'IRBO', 'MTUM', 'DBMF', 'KMLM', 'GLD',
    'SPY', 'QQQ', 'TQQQ', 'BTC-USD', 'SOL-USD', 'ETH-USD', 'BNB-USD',
    'XLE', 'AMLP', 'XLF', 'KRE', 'XLV', 'XBI', 'XLI', 'XLB', 'XLP',
    'XLY', 'VNQ', 'XLRE', 'TLT', 'HYG', 'EFA', 'VGK', '^NSEI',
}

print('Configuration defined ✓')
print(f'  Universe size     : {len(CURRENT_UNIVERSE)} assets')
print(f'  Weak threshold    : Bottom {WEAK_N} by trailing Sharpe')
print(f'  Candidates shown  : Top {TOP_CANDIDATES} per weak stock')
print(f'  Sharpe basis      : {ADPY} days/year')
print(f'  Min Piotroski     : {MIN_PIOTROSKI}/9')
print(f'  Altman safe zone  : > {ALTMAN_SAFE_ZONE}')


# ── Cell 2: Candidate Pool Construction ─────────────────────────────────────
# Hardcoded S&P 500 + NASDAQ 100 tickers with GICS sectors.
# This avoids Wikipedia scraping which is frequently blocked (403 errors).
# Tickers are current as of mid-2025. Update annually if needed.
# Sectors follow the official GICS classification.

# Format: 'TICKER': 'GICS Sector'
CANDIDATE_UNIVERSE_WITH_SECTORS = {
    # ── Information Technology ───────────────────────────────────────────────
    'AAPL':  'Information Technology', 'MSFT':  'Information Technology',
    'NVDA':  'Information Technology', 'AVGO':  'Information Technology',
    'AMD':   'Information Technology', 'INTC':  'Information Technology',
    'QCOM':  'Information Technology', 'TXN':   'Information Technology',
    'MU':    'Information Technology', 'AMAT':  'Information Technology',
    'LRCX':  'Information Technology', 'KLAC':  'Information Technology',
    'MCHP':  'Information Technology', 'ADI':   'Information Technology',
    'CDNS':  'Information Technology', 'SNPS':  'Information Technology',
    'FTNT':  'Information Technology', 'PANW':  'Information Technology',
    'CRWD':  'Information Technology', 'ZBRA':  'Information Technology',
    'HPQ':   'Information Technology', 'GLW':   'Information Technology',
    'STX':   'Information Technology', 'WDC':   'Information Technology',
    'KEYS':  'Information Technology', 'TRMB':  'Information Technology',
    'FFIV':  'Information Technology', 'JNPR':  'Information Technology',
    'AKAM':  'Information Technology', 'VRSN':  'Information Technology',
    'IBM':   'Information Technology', 'ACN':   'Information Technology',
    'INTU':  'Information Technology', 'NOW':   'Information Technology',
    'ADBE':  'Information Technology', 'CRM':   'Information Technology',
    'ORCL':  'Information Technology', 'SAP':   'Information Technology',

    # ── Communication Services ───────────────────────────────────────────────
    'GOOG':  'Communication Services', 'GOOGL': 'Communication Services',
    'META':  'Communication Services', 'NFLX':  'Communication Services',
    'DIS':   'Communication Services', 'CMCSA': 'Communication Services',
    'T':     'Communication Services', 'VZ':    'Communication Services',
    'TMUS':  'Communication Services', 'CHTR':  'Communication Services',
    'EA':    'Communication Services', 'TTWO':  'Communication Services',
    'WBD':   'Communication Services', 'OMC':   'Communication Services',
    'IPG':   'Communication Services', 'FOXA':  'Communication Services',

    # ── Consumer Discretionary ───────────────────────────────────────────────
    'AMZN':  'Consumer Discretionary', 'TSLA':  'Consumer Discretionary',
    'HD':    'Consumer Discretionary', 'MCD':   'Consumer Discretionary',
    'NKE':   'Consumer Discretionary', 'SBUX':  'Consumer Discretionary',
    'LOW':   'Consumer Discretionary', 'TJX':   'Consumer Discretionary',
    'BKNG':  'Consumer Discretionary', 'MAR':   'Consumer Discretionary',
    'HLT':   'Consumer Discretionary', 'YUM':   'Consumer Discretionary',
    'ORLY':  'Consumer Discretionary', 'AZO':   'Consumer Discretionary',
    'ROST':  'Consumer Discretionary', 'DHI':   'Consumer Discretionary',
    'LEN':   'Consumer Discretionary', 'PHM':   'Consumer Discretionary',
    'F':     'Consumer Discretionary', 'GM':    'Consumer Discretionary',
    'EBAY':  'Consumer Discretionary', 'ETSY':  'Consumer Discretionary',
    'BBY':   'Consumer Discretionary', 'DRI':   'Consumer Discretionary',
    'CMG':   'Consumer Discretionary', 'EXPE':  'Consumer Discretionary',
    'ABNB':  'Consumer Discretionary', 'LVS':   'Consumer Discretionary',
    'MGM':   'Consumer Discretionary', 'CCL':   'Consumer Discretionary',

    # ── Consumer Staples ─────────────────────────────────────────────────────
    'WMT':   'Consumer Staples', 'COST':  'Consumer Staples',
    'PG':    'Consumer Staples', 'KO':    'Consumer Staples',
    'PEP':   'Consumer Staples', 'PM':    'Consumer Staples',
    'MO':    'Consumer Staples', 'MDLZ':  'Consumer Staples',
    'CL':    'Consumer Staples', 'KMB':   'Consumer Staples',
    'EL':    'Consumer Staples', 'GIS':   'Consumer Staples',
    'K':     'Consumer Staples', 'HSY':   'Consumer Staples',
    'SJM':   'Consumer Staples', 'CAG':   'Consumer Staples',
    'CPB':   'Consumer Staples', 'HRL':   'Consumer Staples',
    'TSN':   'Consumer Staples', 'MKC':   'Consumer Staples',
    'CLX':   'Consumer Staples', 'CHD':   'Consumer Staples',
    'CVS':   'Consumer Staples', 'WBA':   'Consumer Staples',
    'KR':    'Consumer Staples', 'SYY':   'Consumer Staples',

    # ── Health Care ──────────────────────────────────────────────────────────
    'LLY':   'Health Care', 'JNJ':   'Health Care',
    'UNH':   'Health Care', 'ABBV':  'Health Care',
    'MRK':   'Health Care', 'TMO':   'Health Care',
    'ABT':   'Health Care', 'DHR':   'Health Care',
    'PFE':   'Health Care', 'AMGN':  'Health Care',
    'BSX':   'Health Care', 'SYK':   'Health Care',
    'ISRG':  'Health Care', 'MDT':   'Health Care',
    'EW':    'Health Care', 'BDX':   'Health Care',
    'ZBH':   'Health Care', 'BAX':   'Health Care',
    'HOLX':  'Health Care', 'DXCM':  'Health Care',
    'IDXX':  'Health Care', 'IQV':   'Health Care',
    'A':     'Health Care', 'MTD':   'Health Care',
    'GEHC':  'Health Care', 'VTRS':  'Health Care',
    'BIIB':  'Health Care', 'GILD':  'Health Care',
    'REGN':  'Health Care', 'VRTX':  'Health Care',
    'ALNY':  'Health Care', 'INCY':  'Health Care',
    'EXAS':  'Health Care', 'TECH':  'Health Care',
    'CI':    'Health Care', 'HUM':   'Health Care',
    'CNC':   'Health Care', 'MOH':   'Health Care',

    # ── Financials ───────────────────────────────────────────────────────────
    'BRK-B': 'Financials', 'JPM':   'Financials',
    'BAC':   'Financials', 'WFC':   'Financials',
    'GS':    'Financials', 'MS':    'Financials',
    'BLK':   'Financials', 'SCHW':  'Financials',
    'AXP':   'Financials', 'CB':    'Financials',
    'PGR':   'Financials', 'AFL':   'Financials',
    'MET':   'Financials', 'PRU':   'Financials',
    'TRV':   'Financials', 'ALL':   'Financials',
    'USB':   'Financials', 'PNC':   'Financials',
    'TFC':   'Financials', 'COF':   'Financials',
    'DFS':   'Financials', 'SYF':   'Financials',
    'FITB':  'Financials', 'RF':    'Financials',
    'HBAN':  'Financials', 'CFG':   'Financials',
    'KEY':   'Financials', 'CMA':   'Financials',
    'ZION':  'Financials', 'MTB':   'Financials',
    'ICE':   'Financials', 'CME':   'Financials',
    'CBOE':  'Financials', 'NDAQ':  'Financials',
    'SPGI':  'Financials', 'MCO':   'Financials',
    'V':     'Financials', 'MA':    'Financials',
    'PYPL':  'Financials', 'FIS':   'Financials',
    'FISV':  'Financials', 'GPN':   'Financials',

    # ── Industrials ──────────────────────────────────────────────────────────
    'GE':    'Industrials', 'HON':   'Industrials',
    'CAT':   'Industrials', 'UPS':   'Industrials',
    'RTX':   'Industrials', 'LMT':   'Industrials',
    'BA':    'Industrials', 'NOC':   'Industrials',
    'GD':    'Industrials', 'L3H':   'Industrials',
    'DE':    'Industrials', 'EMR':   'Industrials',
    'ETN':   'Industrials', 'PH':    'Industrials',
    'ROK':   'Industrials', 'AME':   'Industrials',
    'FDX':   'Industrials', 'CSX':   'Industrials',
    'UNP':   'Industrials', 'NSC':   'Industrials',
    'DAL':   'Industrials', 'UAL':   'Industrials',
    'AAL':   'Industrials', 'LUV':   'Industrials',
    'WM':    'Industrials', 'RSG':   'Industrials',
    'CTAS':  'Industrials', 'FAST':  'Industrials',
    'GWW':   'Industrials', 'MSC':   'Industrials',
    'XYL':   'Industrials', 'IEX':   'Industrials',
    'GNRC':  'Industrials', 'CARR':  'Industrials',
    'OTIS':  'Industrials', 'TT':    'Industrials',

    # ── Energy ───────────────────────────────────────────────────────────────
    'XOM':   'Energy', 'CVX':   'Energy',
    'COP':   'Energy', 'EOG':   'Energy',
    'SLB':   'Energy', 'MPC':   'Energy',
    'PSX':   'Energy', 'VLO':   'Energy',
    'PXD':   'Energy', 'OXY':   'Energy',
    'HES':   'Energy', 'DVN':   'Energy',
    'FANG':  'Energy', 'HAL':   'Energy',
    'BKR':   'Energy', 'OKE':   'Energy',
    'WMB':   'Energy', 'KMI':   'Energy',
    'LNG':   'Energy', 'CTRA':  'Energy',

    # ── Materials ────────────────────────────────────────────────────────────
    'LIN':   'Materials', 'APD':   'Materials',
    'SHW':   'Materials', 'FCX':   'Materials',
    'NEM':   'Materials', 'NUE':   'Materials',
    'STLD':  'Materials', 'CF':    'Materials',
    'MOS':   'Materials', 'ALB':   'Materials',
    'PPG':   'Materials', 'ECL':   'Materials',
    'IFF':   'Materials', 'CE':    'Materials',
    'DD':    'Materials', 'DOW':   'Materials',
    'LYB':   'Materials', 'PKG':   'Materials',
    'IP':    'Materials', 'WRK':   'Materials',

    # ── Utilities ────────────────────────────────────────────────────────────
    'NEE':   'Utilities', 'DUK':   'Utilities',
    'SO':    'Utilities', 'D':     'Utilities',
    'AEP':   'Utilities', 'EXC':   'Utilities',
    'SRE':   'Utilities', 'XEL':   'Utilities',
    'PCG':   'Utilities', 'ED':    'Utilities',
    'EIX':   'Utilities', 'ETR':   'Utilities',
    'PEG':   'Utilities', 'FE':    'Utilities',
    'WEC':   'Utilities', 'ES':    'Utilities',
    'AWK':   'Utilities', 'ATO':   'Utilities',
    'LNT':   'Utilities', 'NI':    'Utilities',

    # ── Real Estate ──────────────────────────────────────────────────────────
    'PLD':   'Real Estate', 'AMT':   'Real Estate',
    'CCI':   'Real Estate', 'EQIX':  'Real Estate',
    'PSA':   'Real Estate', 'O':     'Real Estate',
    'WELL':  'Real Estate', 'SPG':   'Real Estate',
    'DLR':   'Real Estate', 'AVB':   'Real Estate',
    'EQR':   'Real Estate', 'INVH':  'Real Estate',
    'MAA':   'Real Estate', 'UDR':   'Real Estate',
    'EXR':   'Real Estate', 'CUBE':  'Real Estate',
    'IRM':   'Real Estate', 'SBAC':  'Real Estate',
    'ARE':   'Real Estate', 'BXP':   'Real Estate',
}

# Try Wikipedia first; fall back silently to the hardcoded dict above.
# This means the notebook works whether Wikipedia is accessible or not.
def get_sp500_tickers():
    try:
        table   = pd.read_html(
            'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', header=0
        )[0]
        tickers = table['Symbol'].str.replace('.', '-', regex=False).tolist()
        sectors = dict(zip(
            table['Symbol'].str.replace('.', '-', regex=False),
            table['GICS Sector']
        ))
        print(f'  S&P 500: {len(tickers)} tickers from Wikipedia ✓')
        return tickers, sectors
    except Exception:
        print('  S&P 500 Wikipedia blocked — using hardcoded list ✓')
        tickers = list(CANDIDATE_UNIVERSE_WITH_SECTORS.keys())
        sectors = CANDIDATE_UNIVERSE_WITH_SECTORS.copy()
        return tickers, sectors


def get_nasdaq100_tickers():
    try:
        tables = pd.read_html('https://en.wikipedia.org/wiki/Nasdaq-100', header=0)
        for t in tables:
            cols = [c.lower() for c in t.columns]
            if 'ticker' in cols or 'symbol' in cols:
                col     = 'Ticker' if 'Ticker' in t.columns else 'Symbol'
                tickers = t[col].str.replace('.', '-', regex=False).tolist()
                print(f'  NASDAQ 100: {len(tickers)} tickers from Wikipedia ✓')
                return tickers
    except Exception:
        pass
    # Fall back: NASDAQ 100 overlap is already covered by the hardcoded dict
    print('  NASDAQ 100 Wikipedia blocked — covered by hardcoded list ✓')
    return []


print('Building candidate pool...')
sp500_tickers, sp500_sectors = get_sp500_tickers()
ndx_tickers                  = get_nasdaq100_tickers()

# Merge Wikipedia results with hardcoded sectors — hardcoded fills any gaps
all_tickers = list(dict.fromkeys(
    sp500_tickers + ndx_tickers + list(CANDIDATE_UNIVERSE_WITH_SECTORS.keys())
))

# Ensure every ticker has a sector entry — hardcoded dict is the fallback
for t, s in CANDIDATE_UNIVERSE_WITH_SECTORS.items():
    if t not in sp500_sectors:
        sp500_sectors[t] = s

# Remove anything already in the current universe
candidate_pool = [t for t in all_tickers if t not in set(CURRENT_UNIVERSE)]

print(f'\nCandidate pool after deduplication and universe exclusion: '
      f'{len(candidate_pool)} tickers across '
      f'{len(set(sp500_sectors.values()))} GICS sectors')


# ── Cell 3: Price Download ───────────────────────────────────────────────────

SCREEN_END   = datetime.today()
SCREEN_START = SCREEN_END - timedelta(days=SHARPE_LOOKBACK_DAYS + 60)
# +60 days buffer for warmup / missing trading days

print(f'\nDownloading prices for current universe...')
print(f'  Period: {SCREEN_START.date()} → {SCREEN_END.date()}')

raw_universe = yf.download(
    CURRENT_UNIVERSE,
    start=SCREEN_START.strftime('%Y-%m-%d'),
    end=SCREEN_END.strftime('%Y-%m-%d'),
    auto_adjust=True,
    progress=False,
)

universe_closes = {}
for ticker in CURRENT_UNIVERSE:
    try:
        s = raw_universe['Close'][ticker].dropna()
        if len(s) >= MIN_TRADING_DAYS:
            universe_closes[ticker] = s
    except Exception:
        pass

print(f'  Universe tickers with sufficient history: {len(universe_closes)}')


# ── Cell 4: Trailing Sharpe for Current Universe ─────────────────────────────
# Uses ADPY=365 to match the main notebook exactly.

def compute_trailing_sharpe(close_series, adpy=ADPY, rf=0.04):
    """
    Trailing Sharpe over the available history in close_series.
    adpy=365 matches the merged-calendar convention in the main notebook.
    """
    rets     = close_series.pct_change().dropna()
    # Use only the last SHARPE_LOOKBACK_DAYS calendar days
    cutoff   = close_series.index[-1] - timedelta(days=SHARPE_LOOKBACK_DAYS)
    rets     = rets[rets.index >= cutoff]
    if len(rets) < MIN_TRADING_DAYS:
        return np.nan
    mean_ret = rets.mean()
    std_ret  = rets.std()
    if std_ret == 0 or np.isnan(std_ret):
        return np.nan
    return (mean_ret * adpy - rf) / (std_ret * np.sqrt(adpy))


print('Computing trailing Sharpe for current universe...')
sharpe_scores = {}
for ticker, close in universe_closes.items():
    sharpe_scores[ticker] = compute_trailing_sharpe(close)

sharpe_series = pd.Series(sharpe_scores).sort_values()

print(f'\n{"Ticker":<12} {"Trailing Sharpe (365-day basis)":>32}')
print('─' * 46)
for ticker, sharpe in sharpe_series.items():
    flag = ' ◄ WEAK' if ticker in sharpe_series.iloc[:WEAK_N].index else ''
    print(f'  {ticker:<10} {sharpe:>10.4f}{flag}')


# ── Cell 5: Identify Weak Assets ─────────────────────────────────────────────

weak_assets = sharpe_series.iloc[:WEAK_N].index.tolist()
weak_equity = [t for t in weak_assets if t not in NON_EQUITY_ASSETS]
weak_non_equity = [t for t in weak_assets if t in NON_EQUITY_ASSETS]

print(f'\n{"═"*60}')
print(f'WEAK ASSETS (Bottom {WEAK_N} by trailing Sharpe)')
print(f'{"═"*60}')
print(f'\n  Individual equities (can be replaced by screened stocks):')
for t in weak_equity:
    print(f'    {t:<10}  Sharpe: {sharpe_series[t]:.4f}')

print(f'\n  ETFs / Crypto / Indices (manual replacement recommended):')
for t in weak_non_equity:
    print(f'    {t:<10}  Sharpe: {sharpe_series[t]:.4f}')
    print(f'            → Replace with a similar ETF in the same category manually')


# ── Cell 6: GICS Sector Lookup for Weak Equities ────────────────────────────
# Uses yfinance info to get the sector for each weak equity and each candidate.
# Results are cached to avoid redundant API calls.

def get_sector(ticker):
    """Fetch GICS sector from yfinance. Returns None if unavailable."""
    try:
        info = yf.Ticker(ticker).info
        return info.get('sector', None)
    except Exception:
        return None


print('Fetching sectors for weak equities...')
weak_equity_sectors = {}
for ticker in weak_equity:
    # First try the S&P 500 Wikipedia table (already downloaded)
    sector = sp500_sectors.get(ticker)
    if not sector:
        sector = get_sector(ticker)
    weak_equity_sectors[ticker] = sector
    print(f'  {ticker:<10}: {sector}')


# ── Cell 7: Fundamental Screening Functions ──────────────────────────────────
# All metrics sourced from yfinance financials where available.
# Missing data is handled gracefully — a metric that cannot be computed
# is skipped rather than causing the entire screen to fail.

def safe_get(d, *keys, default=np.nan):
    """Safely traverse nested dicts / series."""
    val = d
    for k in keys:
        try:
            val = val[k] if isinstance(val, dict) else val.loc[k]
        except (KeyError, TypeError):
            return default
    return val if val is not None else default


def compute_piotroski(ticker):
    """
    Piotroski F-Score (0–9).
    Profitability (4 signals) + Leverage/Liquidity (3) + Efficiency (2).
    Returns (score, detail_dict).
    """
    try:
        t      = yf.Ticker(ticker)
        bs     = t.balance_sheet       # columns = quarters/years newest first
        inc    = t.income_stmt
        cf     = t.cashflow
        info   = t.info

        if bs is None or bs.empty or inc is None or inc.empty:
            return np.nan, {}

        # Helper: get most recent and prior year values
        def yr(df, row, n=0):
            try:
                return float(df.loc[row].iloc[n])
            except Exception:
                return np.nan

        # ── Profitability ────────────────────────────────────────────────────
        net_income   = yr(inc, 'Net Income')
        total_assets = yr(bs,  'Total Assets')
        total_assets_prior = yr(bs, 'Total Assets', 1)
        avg_assets   = np.nanmean([total_assets, total_assets_prior])

        roa          = net_income / avg_assets if avg_assets else np.nan
        op_cf        = yr(cf, 'Operating Cash Flow')
        accruals     = (op_cf - net_income) / avg_assets if avg_assets else np.nan
        roa_prior    = yr(inc, 'Net Income', 1) / total_assets_prior \
                       if total_assets_prior else np.nan

        f1 = int(roa > 0)                        if not np.isnan(roa)      else 0
        f2 = int(op_cf > 0)                      if not np.isnan(op_cf)    else 0
        f3 = int(roa > roa_prior)                if not np.isnan(roa_prior) else 0
        f4 = int(accruals > 0)                   if not np.isnan(accruals) else 0

        # ── Leverage / Liquidity ─────────────────────────────────────────────
        long_debt        = yr(bs, 'Long Term Debt')
        long_debt_prior  = yr(bs, 'Long Term Debt', 1)
        curr_assets      = yr(bs, 'Current Assets')
        curr_liab        = yr(bs, 'Current Liabilities')
        curr_assets_p    = yr(bs, 'Current Assets', 1)
        curr_liab_p      = yr(bs, 'Current Liabilities', 1)

        leverage       = long_debt / avg_assets         if avg_assets else np.nan
        leverage_prior = long_debt_prior / total_assets_prior \
                         if total_assets_prior else np.nan
        curr_ratio       = curr_assets / curr_liab      if curr_liab  else np.nan
        curr_ratio_prior = curr_assets_p / curr_liab_p  if curr_liab_p else np.nan

        # Shares — check for dilution
        shares       = yr(inc, 'Diluted Average Shares')
        shares_prior = yr(inc, 'Diluted Average Shares', 1)

        f5 = int(leverage < leverage_prior)      if not np.isnan(leverage_prior) else 0
        f6 = int(curr_ratio > curr_ratio_prior)  if not np.isnan(curr_ratio_prior) else 0
        f7 = int(shares <= shares_prior)         if not np.isnan(shares_prior)  else 0

        # ── Efficiency ───────────────────────────────────────────────────────
        revenue       = yr(inc, 'Total Revenue')
        revenue_prior = yr(inc, 'Total Revenue', 1)
        gross_profit  = yr(inc, 'Gross Profit')
        gp_prior      = yr(inc, 'Gross Profit', 1)

        gross_margin       = gross_profit / revenue       if revenue       else np.nan
        gross_margin_prior = gp_prior     / revenue_prior if revenue_prior else np.nan
        asset_turnover       = revenue       / avg_assets if avg_assets else np.nan
        asset_turnover_prior = revenue_prior / total_assets_prior \
                               if total_assets_prior else np.nan

        f8 = int(gross_margin > gross_margin_prior) \
             if not np.isnan(gross_margin_prior) else 0
        f9 = int(asset_turnover > asset_turnover_prior) \
             if not np.isnan(asset_turnover_prior) else 0

        score = f1 + f2 + f3 + f4 + f5 + f6 + f7 + f8 + f9
        detail = {
            'F1_ROA_positive':       f1, 'F2_OpCF_positive':    f2,
            'F3_ROA_improving':      f3, 'F4_Accruals_positive': f4,
            'F5_Leverage_falling':   f5, 'F6_Liquidity_rising':  f6,
            'F7_No_dilution':        f7, 'F8_Margin_improving':  f8,
            'F9_Turnover_improving': f9,
        }
        return score, detail

    except Exception as e:
        return np.nan, {'error': str(e)}


def compute_capex_to_ocf(ticker):
    """
    CapEx / Operating Cash Flow.
    Lower is better — high CapEx relative to OCF means the business
    is consuming a lot of cash for maintenance/growth.
    A ratio < 0.5 is generally considered healthy.
    Returns the ratio (positive = CapEx is consuming OCF).
    """
    try:
        cf  = yf.Ticker(ticker).cashflow
        ocf = float(cf.loc['Operating Cash Flow'].iloc[0])
        # CapEx is usually negative in cash flow statements
        capex = float(cf.loc['Capital Expenditure'].iloc[0])
        capex = abs(capex)   # make positive for the ratio
        if ocf <= 0:
            return np.nan    # negative OCF — ratio is not meaningful
        return capex / ocf
    except Exception:
        return np.nan


def compute_altman_z(ticker):
    """
    Altman Z-Score for publicly traded manufacturing firms.
    Z = 1.2*X1 + 1.4*X2 + 3.3*X3 + 0.6*X4 + 1.0*X5
    > 2.99 = safe zone; 1.81–2.99 = grey zone; < 1.81 = distress
    Returns (z_score, distress_flag) where distress_flag=1 means below 1.81.
    """
    try:
        t    = yf.Ticker(ticker)
        bs   = t.balance_sheet
        inc  = t.income_stmt
        info = t.info

        def yr(df, row, n=0):
            try:
                return float(df.loc[row].iloc[n])
            except Exception:
                return np.nan

        total_assets  = yr(bs, 'Total Assets')
        curr_assets   = yr(bs, 'Current Assets')
        curr_liab     = yr(bs, 'Current Liabilities')
        retained_earn = yr(bs, 'Retained Earnings')
        ebit          = yr(inc, 'EBIT')
        revenue       = yr(inc, 'Total Revenue')
        total_liab    = yr(bs, 'Total Liabilities Net Minority Interest')

        market_cap    = info.get('marketCap', np.nan)

        if any(np.isnan(v) for v in [total_assets, curr_assets, curr_liab,
                                      retained_earn, ebit, revenue,
                                      total_liab, market_cap]):
            return np.nan, np.nan

        working_cap = curr_assets - curr_liab

        x1 = working_cap    / total_assets
        x2 = retained_earn  / total_assets
        x3 = ebit           / total_assets
        x4 = market_cap     / total_liab       if total_liab else np.nan
        x5 = revenue        / total_assets

        if np.isnan(x4):
            return np.nan, np.nan

        z = 1.2*x1 + 1.4*x2 + 3.3*x3 + 0.6*x4 + 1.0*x5
        distress_flag = int(z < 1.81)
        return round(z, 3), distress_flag

    except Exception:
        return np.nan, np.nan


def compute_asset_turnover(ticker):
    """
    Asset Turnover = Revenue / Average Total Assets.
    Higher is better — measures how efficiently the firm uses assets to generate revenue.
    """
    try:
        t    = yf.Ticker(ticker)
        bs   = t.balance_sheet
        inc  = t.income_stmt

        def yr(df, row, n=0):
            try:
                return float(df.loc[row].iloc[n])
            except Exception:
                return np.nan

        revenue      = yr(inc, 'Total Revenue')
        assets_now   = yr(bs,  'Total Assets')
        assets_prior = yr(bs,  'Total Assets', 1)
        avg_assets   = np.nanmean([assets_now, assets_prior])

        if np.isnan(avg_assets) or avg_assets == 0:
            return np.nan
        return revenue / avg_assets
    except Exception:
        return np.nan


print('Fundamental screening functions defined ✓')
print('  Metrics: Piotroski F-Score, CapEx/OCF, Asset Turnover, Altman Z-Score')


# ── Cell 8: Screen Candidate Pool ────────────────────────────────────────────
# This cell does the heavy lifting — it downloads price history and
# fundamental data for every candidate, computes all metrics, and
# filters to a clean shortlist.
#
# Runtime note: fetching fundamentals from yfinance for hundreds of tickers
# is slow (~1-2 seconds per ticker). For the full S&P500+NDX pool this can
# take 15-30 minutes. A progress counter is printed every 25 tickers.
# Results are cached in candidate_results_df so you only need to run once.

def screen_candidate(ticker, sector_filter=None):
    """
    Run the full screen on a single candidate ticker.
    Returns a dict of all metrics, or None if the ticker fails basic checks.
    sector_filter: if provided, skip tickers not in this sector.
    """
    try:
        # ── Sector check ─────────────────────────────────────────────────────
        sector = sp500_sectors.get(ticker) or get_sector(ticker)
        if sector_filter and sector != sector_filter:
            return None

        # ── Price / liquidity check ───────────────────────────────────────────
        raw = yf.download(
            ticker,
            start=SCREEN_START.strftime('%Y-%m-%d'),
            end=SCREEN_END.strftime('%Y-%m-%d'),
            auto_adjust=True,
            progress=False,
        )
        if raw.empty or len(raw) < MIN_TRADING_DAYS:
            return None

        close  = raw['Close'].squeeze()
        volume = raw['Volume'].squeeze()

        # Liquidity filter
        avg_dollar_vol = (close * volume).tail(60).mean()
        if avg_dollar_vol < MIN_AVG_DOLLAR_VOL:
            return None

        # Trailing Sharpe
        sharpe = compute_trailing_sharpe(close)
        if np.isnan(sharpe):
            return None

        # ── Momentum filter: price must be above its 200-day MA ──────────────
        if len(close) >= 200:
            ma200 = close.rolling(200).mean().iloc[-1]
            above_ma200 = close.iloc[-1] > ma200
        else:
            above_ma200 = False

        # ── Fundamentals ──────────────────────────────────────────────────────
        piotroski, _  = compute_piotroski(ticker)
        capex_ocf     = compute_capex_to_ocf(ticker)
        asset_turn    = compute_asset_turnover(ticker)
        altman_z, distress_flag = compute_altman_z(ticker)

        return {
            'ticker'         : ticker,
            'sector'         : sector,
            'sharpe_365'     : round(sharpe, 4),
            'above_ma200'    : above_ma200,
            'piotroski'      : piotroski,
            'capex_to_ocf'   : round(capex_ocf, 4)   if not np.isnan(capex_ocf)  else np.nan,
            'asset_turnover' : round(asset_turn, 4)  if not np.isnan(asset_turn) else np.nan,
            'altman_z'       : altman_z,
            'distress_flag'  : distress_flag,
            'avg_dollar_vol' : round(avg_dollar_vol / 1e6, 2),  # in $M
        }

    except Exception as e:
        return None


# ── Run the screen sector-by-sector for weak equities only ───────────────────
# We only screen candidates whose sector matches a weak equity's sector.
# This avoids screening the entire S&P500+NDX unnecessarily.

required_sectors = set(weak_equity_sectors.values()) - {None}
print(f'Sectors requiring replacement candidates: {required_sectors}')
print(f'Screening {len(candidate_pool)} candidates '
      f'(filtered to {len(required_sectors)} sectors)...\n')

candidate_results = []
screened = 0

for ticker in candidate_pool:
    # Quick sector pre-filter using the Wikipedia table before downloading
    ticker_sector = sp500_sectors.get(ticker)
    if ticker_sector and ticker_sector not in required_sectors:
        continue   # skip — wrong sector, no point downloading

    result = screen_candidate(ticker)
    if result is not None:
        candidate_results.append(result)

    screened += 1
    if screened % 25 == 0:
        print(f'  Screened {screened} tickers, '
              f'{len(candidate_results)} passed basic checks so far...')

candidate_results_df = pd.DataFrame(candidate_results)

if candidate_results_df.empty:
    print('No candidates passed the basic screen.')
else:
    print(f'\nCandidates passing basic screen: {len(candidate_results_df)}')
    print(candidate_results_df[[
        'ticker','sector','sharpe_365','above_ma200',
        'piotroski','capex_to_ocf','asset_turnover',
        'altman_z','distress_flag'
    ]].to_string(index=False))


# ── Cell 9: Apply Fundamental Filters & Score ────────────────────────────────
# Filter the candidate pool to fundamentally healthy stocks, then rank by
# a composite score for each sector.

def apply_fundamental_filters(df):
    """
    Hard filters — a candidate must pass ALL of these to be considered:
      1. Piotroski F-Score >= MIN_PIOTROSKI (default 5)
      2. Altman distress_flag == 0 (not in distress zone)
      3. Price above 200-day MA (confirmed uptrend)
      4. CapEx/OCF < 0.8 (not burning through cash)
    """
    mask = pd.Series(True, index=df.index)

    if 'piotroski' in df.columns:
        mask &= df['piotroski'].fillna(0) >= MIN_PIOTROSKI

    if 'distress_flag' in df.columns:
        mask &= df['distress_flag'].fillna(1) == 0

    if 'above_ma200' in df.columns:
        mask &= df['above_ma200'] == True

    if 'capex_to_ocf' in df.columns:
        mask &= df['capex_to_ocf'].fillna(999) < 0.8

    return df[mask].copy()


def compute_composite_score(df):
    """
    Composite score (higher = better) combining:
      - Sharpe (40% weight)      : momentum quality
      - Piotroski (25% weight)   : financial health
      - Asset Turnover (20%)     : operational efficiency
      - Altman Z-Score (15%)     : solvency buffer

    Each metric is rank-normalised to [0,1] within the candidate pool
    before weighting so no single metric dominates due to scale.
    """
    scored = df.copy()

    def rank_norm(series):
        """Rank-normalise a series to [0, 1]. NaN → 0."""
        filled = series.fillna(series.min())
        ranked = filled.rank(pct=True)
        return ranked

    scored['score_sharpe']   = rank_norm(scored['sharpe_365'])
    scored['score_piotroski']= rank_norm(scored['piotroski'].fillna(0))
    scored['score_turnover'] = rank_norm(scored['asset_turnover'].fillna(0))
    scored['score_altman']   = rank_norm(scored['altman_z'].fillna(0))

    scored['composite_score'] = (
        0.40 * scored['score_sharpe']    +
        0.25 * scored['score_piotroski'] +
        0.20 * scored['score_turnover']  +
        0.15 * scored['score_altman']
    )

    return scored.sort_values('composite_score', ascending=False)


if candidate_results_df.empty:
    filtered_df = pd.DataFrame()
    print('No candidates to filter.')
else:
    filtered_df = apply_fundamental_filters(candidate_results_df)
    filtered_df = compute_composite_score(filtered_df)

    print(f'Candidates after fundamental filters: {len(filtered_df)}')
    print(f'\nFilters applied:')
    print(f'  Piotroski >= {MIN_PIOTROSKI}')
    print(f'  Altman Z not in distress zone (Z > 1.81)')
    print(f'  Price above 200-day MA')
    print(f'  CapEx/OCF < 0.80')


# ── Cell 10: Surface Top Candidates Per Weak Equity ──────────────────────────
# For each weak individual equity, find the top N candidates in the same
# GICS sector, ranked by composite score.

print('\n' + '═'*70)
print('REPLACEMENT CANDIDATES — TOP 5 PER WEAK EQUITY')
print('Ranked by composite score (Sharpe 40% | Piotroski 25% | '
      'Asset Turnover 20% | Altman Z 15%)')
print('Sharpe quoted on 365-day basis to match main notebook')
print('═'*70)

display_cols = [
    'ticker', 'sector', 'sharpe_365', 'piotroski',
    'capex_to_ocf', 'asset_turnover', 'altman_z', 'composite_score',
    'avg_dollar_vol'
]

swap_recommendations = {}

for weak_ticker in weak_equity:
    target_sector = weak_equity_sectors.get(weak_ticker)
    weak_sharpe   = sharpe_series.get(weak_ticker, np.nan)

    print(f'\n{"─"*70}')
    print(f'  WEAK: {weak_ticker}')
    print(f'  Trailing Sharpe : {weak_sharpe:.4f}')
    print(f'  Sector          : {target_sector}')
    print(f'{"─"*70}')

    if filtered_df.empty or target_sector is None:
        print('  No candidates available for this sector.')
        swap_recommendations[weak_ticker] = []
        continue

    sector_candidates = filtered_df[
        filtered_df['sector'] == target_sector
    ].head(TOP_CANDIDATES)

    if sector_candidates.empty:
        print(f'  No filtered candidates found in sector: {target_sector}')
        print(f'  Tip: consider a same-sector ETF as a manual replacement.')
        swap_recommendations[weak_ticker] = []
    else:
        cols_present = [c for c in display_cols if c in sector_candidates.columns]
        print(sector_candidates[cols_present].to_string(index=False))
        swap_recommendations[weak_ticker] = sector_candidates['ticker'].tolist()

print(f'\n{"═"*70}')
print('SUMMARY OF RECOMMENDED SWAPS')
print('═'*70)
for weak_ticker, candidates in swap_recommendations.items():
    if candidates:
        print(f'  Replace {weak_ticker:<8} → candidates: {", ".join(candidates)}')
    else:
        print(f'  Replace {weak_ticker:<8} → no candidates found (manual review needed)')


# ── Cell 11: Proposed New Universe ───────────────────────────────────────────
# Automatically select the top-1 candidate per weak equity to show
# what the universe would look like after swaps.
# You can override any swap manually before running the main notebook.

print('\n' + '═'*70)
print('PROPOSED UPDATED UNIVERSE')
print('(Top-1 candidate auto-selected; override manually as needed)')
print('═'*70)

updated_universe = list(CURRENT_UNIVERSE)  # copy

swap_log = []
for weak_ticker, candidates in swap_recommendations.items():
    if candidates:
        replacement = candidates[0]  # top-ranked by composite score
        idx = updated_universe.index(weak_ticker)
        updated_universe[idx] = replacement
        swap_log.append((weak_ticker, replacement,
                         sharpe_series.get(weak_ticker, np.nan),
                         filtered_df.loc[
                             filtered_df['ticker'] == replacement,
                             'sharpe_365'
                         ].values[0] if not filtered_df.empty else np.nan))
        print(f'  OUT: {weak_ticker:<10}  →  IN: {replacement:<10}  '
              f'(Sharpe: {sharpe_series.get(weak_ticker, np.nan):.3f} → '
              f'{filtered_df.loc[filtered_df["ticker"]==replacement, "sharpe_365"].values[0]:.3f})')
    else:
        print(f'  KEEP: {weak_ticker:<10}  (no replacement found)')

print(f'\nProposed universe ({len(updated_universe)} assets):')
print(updated_universe)

swap_log_df = pd.DataFrame(
    swap_log,
    columns=['removed', 'added', 'old_sharpe', 'new_sharpe']
)
print('\nSwap log:')
print(swap_log_df.to_string(index=False))


# ── Cell 12: Schedule & Next Steps ───────────────────────────────────────────

print('\n' + '═'*70)
print('NEXT STEPS')
print('═'*70)
print("""
  1. Review the Top 5 candidates for each weak asset in Cell 10.
     Override auto-selections in Cell 11 if preferred.

  2. Copy the updated_universe list into TICKERS / BEST_LOGIC in Cell 1
     of the main notebook.

  3. For any new equity added, add a signal config entry to build_signals()
     in Cell 4 of the main notebook. Suggested starting config:
       'NEW_TICKER': [('MACD(9,26,9)', lambda: macd_signal(c, 9, 26, 9)),
                      ('KAMA(10,2,30)', lambda: kama_signal(c, 10, 2, 30))]
     Then run a parameter optimisation pass (Cell 4 logic) before going live.

  4. Re-run this refresher notebook annually, or whenever a position's
     trailing Sharpe drops below the weak threshold mid-year.

  5. ETF/crypto/index slots flagged as weak (Cell 5) require manual review —
     find a similar-category ETF with better recent performance and update
     BEST_LOGIC accordingly.
""")

next_rescreen = datetime.today() + relativedelta(years=1)
print(f'  Next scheduled rescreen: {next_rescreen.strftime("%Y-%m-%d")}')
print('═'*70)

Configuration defined ✓
  Universe size     : 45 assets
  Weak threshold    : Bottom 10 by trailing Sharpe
  Candidates shown  : Top 5 per weak stock
  Sharpe basis      : 365 days/year
  Min Piotroski     : 5/9
  Altman safe zone  : > 2.99
Building candidate pool...
  S&P 500 Wikipedia blocked — using hardcoded list ✓
  NASDAQ 100 Wikipedia blocked — covered by hardcoded list ✓

Candidate pool after deduplication and universe exclusion: 303 tickers across 11 GICS sectors

  Period: 2025-04-03 → 2026-06-02
  Universe tickers with sufficient history: 45
Computing trailing Sharpe for current universe...

Ticker        Trailing Sharpe (365-day basis)
──────────────────────────────────────────────
  BTC-USD       -0.7898 ◄ WEAK
  ^NSEI         -0.7245 ◄ WEAK
  SOL-USD       -0.6199 ◄ WEAK
  ETH-USD       -0.0737 ◄ WEAK
  XLP           -0.0186 ◄ WEAK
  MSFT           0.0548 ◄ WEAK
  XLF            0.0613 ◄ WEAK
  TLT            0.1988 ◄ WEAK
  BNB-USD        0.2801 ◄ WEAK
  XLRE           0

$JNPR: possibly delisted; no timezone found

1 Failed download:
['JNPR']: possibly delisted; no timezone found


  Screened 25 tickers, 24 passed basic checks so far...

Candidates passing basic screen: 34
ticker                 sector  sharpe_365  above_ma200  piotroski  capex_to_ocf  asset_turnover  altman_z  distress_flag
  AVGO Information Technology      2.0118         True          8        0.0226          0.3794   16.2800         0.0000
   AMD Information Technology      3.1788         True          8        0.1263          0.4740   37.6660         0.0000
  INTC Information Technology      3.2726         True          6        1.5104          0.2591    4.6240         0.0000
  QCOM Information Technology      1.4665         True          6        0.0851          0.8411    7.9980         0.0000
   TXN Information Technology      1.6912         True          7        0.6361          0.5045   12.7780         0.0000
    MU Information Technology      4.7365         True          7        0.9048          0.4911   27.0750         0.0000
  AMAT Information Technology      3.0647         True      